In [1]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
from scipy.io import wavfile
import itertools
import io
import numpy as np
import json
import re
import zipfile
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [3]:
# from huggingface_hub import snapshot_download
# import time

# while True:
#     try:
#         snapshot_download(
#             repo_id="joujiboi/japanese-anime-speech-v2", 
#             repo_type="dataset", 
#             local_dir="./japanese-anime-speech-v2",
#         )
#         break
#     except Exception as e:
#         time.sleep(60 * 5)

In [7]:
files = glob('japanese-anime-speech-v2/*/*.parquet')
files = [f for f in files if 'train' not in f]
len(files)

45

In [10]:
def loop(files):

    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['OPENBLAS_NUM_THREADS'] = '1'
    
    files, _ = files

    data = []
    for f in tqdm(files):
        base = '_'.join(f.split('/')[:2]) + '_audio'
        f_new = f.replace('/', '-').replace('.parquet', '')
        os.makedirs(base, exist_ok=True)
        df = pd.read_parquet(f)
        for i in range(len(df)):
            try:
                t = df['transcription'].iloc[i].strip()
                if len(t) < 2:
                    continue
                audio_filename = f'{f_new}_{i}.mp3'
                audio_filename = os.path.join(base, audio_filename)
                b = df['audio'].iloc[i]['bytes']
                audio_np, sr = sf.read(io.BytesIO(b))
                if audio_np.ndim > 1:
                    audio_np = audio_np.mean(axis=1)
                if audio_np.shape[0] < 10000:
                    continue
                sf.write(audio_filename, audio_np, sr)
                
                data.append({
                    'audio_filename': audio_filename,
                    'text': t,
                    'speaker': f"{base}"
                })
            except Exception as e:
                pass
        
    return data

In [11]:
data = multiprocessing(files, loop, cores = 20)

100%|██████████| 2/2 [18:09<00:00, 544.85s/it]


In [12]:
len(data)

290543

In [13]:
from datasets import Dataset

dataset = Dataset.from_list(data)
dataset[0]

{'audio_filename': 'japanese-anime-speech-v2_data_audio/japanese-anime-speech-v2-data-sfw-00002-of-00039_0.mp3',
 'text': 'ラジャりました！',
 'speaker': 'japanese-anime-speech-v2_data_audio'}

In [14]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'japanese-anime-speech-v2')

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 10.02ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1): 100%|█████████▉| 13.7MB / 13.8MB, 1.35MB/s  
Processing Files (1 / 1): 100%|██████████| 13.8MB / 13.8MB, 1.35MB/s  
Processing Files (1 / 1): 100%|██████████| 13.8MB / 13.8MB, 1.38MB/s  
New Data Upload: 100%|██████████| 13.8MB / 13.8MB, 1.38MB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:13<00:00, 13.52s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/c81efdfa706f97794507e94b867e9901354a0810', commit_message='Upload dataset', commit_description='', oid='c81efdfa706f97794507e94b867e9901354a0810', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)

In [16]:
audio_files = [d['audio_filename'] for d in data]

a = list(set(audio_files))
with open('japanese-anime-speech-v2-audio.json', 'w') as fopen:
    json.dump(a, fopen)

pd.DataFrame({'audio': a}).to_parquet('japanese-anime-speech-v2-audio.parquet')

In [3]:
# !zip -rq japanese-anime-speech-v2_data_audio.zip japanese-anime-speech-v2_data_audio

In [4]:
# !hf upload malaysia-ai/Multilingual-TTS japanese-anime-speech-v2_data_audio.zip --repo-type=dataset

In [7]:
# !zip -rq japanese-anime-speech-v2_data_audio_neucodec.zip japanese-anime-speech-v2_data_audio_neucodec

In [8]:
# !hf upload malaysia-ai/Multilingual-TTS japanese-anime-speech-v2_data_audio_neucodec.zip --repo-type=dataset